<strong># OVERVIEW OF THE STUDY</strong>
<br><br>
<strong>## Parameters of the absorber-location sweep</strong>
<br><br>
<strong>Material A refractive index</strong><br>
n<sub>A</sub> ∈ [1.1, 2.9] (step 0.2)
<br><br>
<strong>Material B refractive index</strong><br>
n<sub>B</sub> = 1.0
<br><br>
<strong>Absorber extinction coefficient</strong><br>
k ∈ {0.05, 0.10, 0.20, 0.40, 0.60, 0.80, 1.00}
<br><br>
<strong>Absorber location</strong><br>
(i) Core only<br>
(ii) Low-index layers only (material B)<br>
(iii) All Bragg layers (materials A and B)<br>
(iv) Additional outer absorbing shell
<br><br>
<strong>Design wavelength</strong><br>
λ<sub>design</sub> ∈ {480, 560, 640} nm
<br><br>
<strong>Number of Bragg layers</strong><br>
N = 60
<br><br>
<strong>Outer Bragg layer</strong><br>
Material A
<br><br>
<strong>Collection geometry</strong><br>
Backward hemisphere collection
<br><br>
<strong>Collection numerical aperture</strong><br>
NA = 1.0
<br><br>
<strong>Objective</strong><br>
Quantify how absorber placement influences the optical response, colour quality, scattering efficiency, and collection efficiency of Bragg onion photonic pigments while keeping the absorber strength, geometry, and collection conditions fixed.


## Quantities to analyze


### 1. Colour performance

Rosch–MacAdam colour solid

Optimal colour contour

CIELAB a*–b*

Colour strips

η_C, η_L heatmaps

### 2. Scattering

Differential scattering vs θ (dB)

Integrated efficiencies vs λ

Asymmetry g, albedo

Forward / side / backward (dB)

Pie chart at α = 70.5°

### 3. Near fields

|E| (dB), xz / yz / xy

Poynting flow + streamlines

Hotspots

### 4. Resonances

Mie l‑decomposition

l‑resolved spectra

### 5. Sensitivity

η_C, η_L vs n_layers, Δn, k

In [ ]:
# CELL 0 — IMPORTS & GLOBAL SETTINGS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from bragg_onion._notebook_tools.io import load_sweep_outputs
from bragg_onion._notebook_tools.colour import add_display_normalized_colour_columns
from bragg_onion._notebook_tools.strip_plots import (
    select_best_etaC_by_wavelength,
    plot_best_etaC_colour_strip,
    select_strip_grid,
    plot_colour_strip_grid,
    plot_sweep_heatmap_local,
    prepare_strip_grid,
    summarize_strip_grid_cases,
    plot_colour_strip_grid,
)
from bragg_onion._notebook_tools.rosch_plots import (
    make_rosch_macadam_figures,
    add_highlight_cielab_points,
)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 600

print("Notebook helper modules imported successfully.")

In [ ]:
# CELL 1 — PARAMETER SPACE DEFINITION

n_A_vals = np.arange(1.1, 3.0, 0.2)
k_vals = np.array([0.05, 0.10, 0.20, 0.40, 0.60, 0.80, 1.00])
n_B = 1.0

design_wavelengths_nm = np.array([480, 560, 640])
n_layers_vals = [60]
half_angles_deg = np.array([90])
collection_NA_vals = np.sin(np.deg2rad(half_angles_deg))

print("n_A_vals:", n_A_vals)
print("k_vals:", k_vals)
print("design_wavelengths_nm:", design_wavelengths_nm)
print("n_layers_vals:", n_layers_vals)
print("half_angles_deg:", half_angles_deg)

In [ ]:
# CELL 2 — MASTER PARAMETER GRID (PRIMARY SWEEP)

PARAMETER_GRID = {
    "peak_wavelength_m": design_wavelengths_nm * 1e-9,
    "n_layers": n_layers_vals,
    "outer_layer": ["A"],
    "core_thickness_factor": 0.5,
}

PARAMETER_GRID

In [ ]:
# CELL 3 — OUTPUT SOURCE / NOTEBOOK SETTINGS

# Windows path to the copied workstation output directory.
# This can stay in Windows form even when the notebook runs in WSL:
# load_sweep_outputs(...) will convert it automatically if needed.
SOURCE_OUTPUT_DIR_WINDOWS = (
    r"C:\Users\SchwarzN\OneDrive - Université de Fribourg"
    r"\Institution\P1_BraggSpericalPigments\Simulation\MIE_Computation"
    r"\Scattnlay_BraggOnion\outputs\analyse_absorber_location"
)

# Default colour view for object-colour-style plotting:
DEFAULT_COLOUR_VIEW = "display"   # "raw", "display", or "both"

print("SOURCE_OUTPUT_DIR_WINDOWS:")
print(SOURCE_OUTPUT_DIR_WINDOWS)
print("DEFAULT_COLOUR_VIEW:", DEFAULT_COLOUR_VIEW)

In [ ]:
# CELL 4 — LOAD SWEEP OUTPUTS & ADD DISPLAY-NORMALIZED COLOURS

df_all, OUTPUT_DIR, USE_PARQUET = load_sweep_outputs(
    SOURCE_OUTPUT_DIR_WINDOWS,
    combine_if_needed=True,
    verbose=True,
)

# Working dataframe for plotting/analysis
df_plot = add_display_normalized_colour_columns(df_all, inplace=False)

print("\nLoaded dataframe summary:")
print("OUTPUT_DIR   :", OUTPUT_DIR)
print("USE_PARQUET  :", USE_PARQUET)
print("df_all rows  :", len(df_all))
print("df_plot rows :", len(df_plot))

display(df_plot.head())

In [ ]:
print(df_all.columns)

In [ ]:
print(df_plot.shape)

if "absorber_location" in df_plot.columns:
    print(df_plot["absorber_location"].value_counts())
else:
    print("absorber_location column missing")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameter values
k_values = np.sort(df_all['k'].unique())
nA_values = np.sort(df_all['n_A'].unique())
layer_values = np.sort(df_all['n_layers'].unique())
print(df_all['design_peak_wavelength_nm'].unique())

# Marker for each absorber location
absorber_locations = np.sort(df_all["absorber_location"].unique())

marker_dict = {
    "core": "o",
    "low_index": "s",
    "all_layers": "^",
    "outer_shell": "D",
}

# Small x-offset for each n_A
offsets = np.linspace(-8, 8, len(nA_values))
offset_dict = dict(zip(nA_values, offsets))

# Marker size for each k value
sizes = np.linspace(60, 180, len(k_values))
size_dict = dict(zip(k_values, sizes))

plt.figure(figsize=(10, 6))

for k in k_values:
    if k != 0.05:
        continue

    for nA in nA_values:
        # if nA == 1.5:
        #     continue

        for absorber_location in absorber_locations:
            if absorber_location != "outer_shell": # 'core' 'low_index' 'all_layers' 'outer_shell'
                continue

            subset = df_all[
                (df_all['collection_na'] == 1.0) &
                (df_all['k'] == k) &
                (df_all['n_A'] == nA) &
                (df_all['absorber_location'] == absorber_location)
            ]

            if subset.empty:
                continue

            hue = np.mod(np.degrees(np.arctan2(subset['b'], subset['a'])), 360)
            # Shift hsv colour map
            hue_shift = 290  # degrees

            hue_display = (hue + hue_shift) % 360
            
            plt.scatter(
                subset['design_peak_wavelength_nm'] + offset_dict[nA],
                hue,
                # c=hue_display,
                c=subset['hex'],
                cmap='hsv',
                # marker=marker_dict[subset['absorber_location'].iloc[0]],
                marker=marker_dict[absorber_location],
                s=size_dict[k],
                edgecolor=None,
                label=f'k={k}, nA={nA}, absorber_location={absorber_location}',
                alpha=1
            )

import matplotlib as mpl

ax = plt.gca()      # or better: fig, ax = plt.subplots(...)

norm = mpl.colors.Normalize(vmin=0, vmax=360)
sm = mpl.cm.ScalarMappable(norm=norm, cmap='hsv')
sm.set_array([])
ticks = np.linspace(0, 360, 7)
cbar = plt.colorbar(sm, ax=ax, ticks=ticks)
cbar.set_label("Hue angle [degrees] (hsv colormap)")

plt.xticks(df_all['design_peak_wavelength_nm'].unique())
plt.xlabel('Design Peak Wavelength (nm)')
plt.yticks(np.arange(0, 361, 60))
plt.ylabel('Hue Angle [degrees] (CIE a*b* plane)')
plt.title('Hue Angle vs Design Peak Wavelength')
from matplotlib.lines import Line2D

size_handles = []

for k in k_values:
    size_handles.append(
        Line2D(
            [0], [0],
            marker='o',
            color='w',
            markerfacecolor='gray',
            markeredgecolor='k',
            markersize=np.sqrt(size_dict[k]),
            label=f'k = {k:.2f}'
        )
    )

shape_handles = []

for loc in absorber_locations:
    shape_handles.append(
        Line2D(
            [0], [0],
            marker=marker_dict[loc],
            linestyle='None',
            color='k',
            markersize=8,
            label=loc.replace('_', ' ')
        )
    )

leg1 = plt.legend(
    handles=size_handles,
    title='Extinction k',
    loc='upper right'
)

plt.gca().add_artist(leg1)

plt.legend(
    handles=shape_handles,
    title='Absorber location',
    loc='lower right'
)

plt.text(
    0.02, 0.2,
    "Horizontal offset indicates $n_A$\n"
    "Marker size indicates $k$\n"
    "Marker shape indicates absorber location",
    transform=plt.gca().transAxes,
    va='top',
    fontsize=10,
    bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray')
)

# possible target hues
target_hues = np.array([30, 60, 90, 150, 210, 270, 315])
# corresponding target wavelengths
target_wavelengths = np.array([760, 680, 620, 540, 500, 440, 380])
for target_hue in target_hues:
    plt.axhline(
        y=target_hue,
        color='gray',
        linestyle='--',
        linewidth=0.8,
        alpha=0.5
    )

plt.show()

In [ ]:
# CELL 5 — MAIN PLOT
# PART A — ROSCH–MACADAM COLOUR SOLID
# ============================================================

ROSCH_COLOUR_VIEW = "both"   # "raw", "display", or "both"

generated_figures = make_rosch_macadam_figures(
    df_plot,
    colour_view=ROSCH_COLOUR_VIEW,
    max_overlay_points=2000,
    fixed_marker_size=4.0,
    marker_symbol_variable="n_layers",   # change freely
    marker_symbol_style="solid_and_line",  # "solid_only" or "solid_and_line"
    marker_edge_color="black",           # or None
    marker_edge_width=0.4,
    optimal_marker_size=4.8,
    optimal_marker_edge_color=None,
    optimal_marker_edge_width=0.7,
    save_html=True,
    output_dir=OUTPUT_DIR,
    show_figures=True,
)

print("Generated Rosch–MacAdam figure keys:", list(generated_figures.keys()))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

group_cols = [
    "absorber_location",
    "design_peak_wavelength_nm",
]

idx = (
    df_all
    .groupby(group_cols)["eta_C"]
    .idxmax()
)

df_best = df_all.loc[idx].copy()

locations = [
    "core",
    "low_index",
    "all_layers",
    "outer_shell",
]

wavelengths = sorted(df_best["design_peak_wavelength_nm"].unique())

fig, ax = plt.subplots(
    figsize=(3.0 * len(wavelengths),
             2.3 * len(locations))
)

for i, loc in enumerate(locations):
    for j, wl in enumerate(wavelengths):

        row = df_best[
            (df_best["absorber_location"] == loc)
            & (df_best["design_peak_wavelength_nm"] == wl)
        ]

        if row.empty:
            continue

        row = row.iloc[0]

        rect = Rectangle(
            (j, len(locations)-1-i),
            1,
            1,
            facecolor=row["hex"],
            edgecolor="black",
        )
        print(row["hex"])

        ax.add_patch(rect)

        txt = (
            f"ηC={100*row['eta_C']:.0f}%\n"
            f"ηL={100*row['eta_L']:.0f}%\n"
            f"nA={row['n_A']:.1f}\n"
            f"k={row['k']:.2f}"
        )

        ax.text(
            j + 0.5,
            len(locations)-1-i + 0.5,
            txt,
            ha="center",
            va="center",
            fontsize=9,
            color="black",
        )

ax.set_xlim(0, len(wavelengths))
ax.set_ylim(0, len(locations))

ax.set_xticks(np.arange(len(wavelengths)) + 0.5)
ax.set_xticklabels([f"{w:.0f} nm" for w in wavelengths])

ax.set_yticks(np.arange(len(locations)) + 0.5)
ax.set_yticklabels(locations[::-1])

ax.set_title("Best ηC case for each absorber location and target wavelength")

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

heatmap_data = (
    df_best
    .pivot(
        index="absorber_location",
        columns="design_peak_wavelength_nm",
        values="eta_C",
    )
)

plt.figure(figsize=(7,4))

sns.heatmap(
    100 * heatmap_data,
    annot=True,
    fmt=".0f",
    cmap="viridis",
    cbar_kws={"label": "ηC [%]"},
)

plt.xlabel("Design wavelength (nm)")
plt.ylabel("Absorber location")
plt.title("Maximum chroma achieved by each absorber location")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

marker_dict = {
    "core": "o",
    "low_index": "s",
    "all_layers": "^",
    "outer_shell": "D",
}

size_dict = {
    k: s
    for k, s in zip(
        sorted(df_all["k"].unique()),
        np.linspace(50, 250, df_all["k"].nunique())
    )
}

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15,5),
    sharex=True,
    sharey=True,
)

for ax, wl in zip(
    axes,
    sorted(df_all["design_peak_wavelength_nm"].unique())
):

    subset_wl = df_all[
        df_all["design_peak_wavelength_nm"] == wl
    ]

    for loc in subset_wl["absorber_location"].unique():

        sub = subset_wl[
            subset_wl["absorber_location"] == loc
        ]

        ax.scatter(
            100 * sub["eta_L"],
            100 * sub["eta_C"],
            c=sub["hex"],
            s=[size_dict[v] for v in sub["k"]],
            marker=marker_dict[loc],
            edgecolors="k",
            alpha=0.8,
            label=loc,
        )

    ax.set_title(f"{wl:.0f} nm")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("ηC [%]")

for ax in axes:
    ax.set_xlabel("ηL [%]")

handles = []

for loc, marker in marker_dict.items():
    handles.append(
        plt.Line2D(
            [0],
            [0],
            marker=marker,
            color="k",
            linestyle="None",
            markersize=8,
            label=loc,
        )
    )

fig.legend(
    handles=handles,
    loc="lower center",
    ncols=2,
    bbox_to_anchor=(0.5, 0.7)
)

fig.suptitle(
    "Chroma–lightness tradeoff for different absorber placements",
    y=1.05
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PART B — COLOUR STRIP OPTION 1
# Best η_C case for each design wavelength
# ============================================================

fixed_filters_best_strip = {
    # "collection_angle_deg": 90.0,
    # "n_layers": 100,
    # "n_A": 2.1,
    "k": 0.05,
}

df_best_strip = select_best_etaC_by_wavelength(
    df_plot,   # use df_plot if you want hex_display
    fixed_filters=fixed_filters_best_strip,
    x="design_peak_wavelength_nm",
    score_col="eta_C",
)

plot_best_etaC_colour_strip(
    df_best_strip,
    x="design_peak_wavelength_nm",
    colour_hex_col="hex",   # or "hex"
    annotate_cols=["eta_C", "eta_L"],
    title="Best η_C case for each design wavelength\n(fixed k = 0.05)",
)

plt.show()

# Print properties of the best strip dataframe for reference:
df_best_strip_print = df_best_strip.copy().sort_values("design_peak_wavelength_nm")

if "eta_C" in df_best_strip_print.columns:
    df_best_strip_print["eta_C_pct"] = (100 * df_best_strip_print["eta_C"]).round(0).astype(int)

if "eta_L" in df_best_strip_print.columns:
    df_best_strip_print["eta_L_pct"] = (100 * df_best_strip_print["eta_L"]).round(0).astype(int)

cols_to_show = [
    "design_peak_wavelength_nm",
    "case_index",
    "eta_C_pct",
    "eta_L_pct",
    "n_layers",
    "n_A",
    "k",
    "collection_angle_deg",
    "outer_layer",
    "hex_display",
]

cols_to_show = [c for c in cols_to_show if c in df_best_strip_print.columns]

display(df_best_strip_print[cols_to_show])

In [ ]:
# ============================================================
# PART C — COLOUR STRIP OPTION 2
# Multi-row strip grid:
# rows = full range of one sweep parameter
# columns = design wavelength
# ============================================================


# ------------------------------------------------------------
# User choices
# ------------------------------------------------------------
row_variable = "n_A"   # e.g. "k", "n_layers", "collection_angle_deg", "n_A"
colour_hex_col = "hex"                  # or "hex_display"

fixed_filters_grid = {
    "collection_angle_deg": 90,   # removed automatically if row_variable matches
    # "n_layers": 100,
    "n_A": 2.1,
    "k": 0.05,
}

# ------------------------------------------------------------
# Prepare data + auto-generated metadata
# ------------------------------------------------------------
df_grid, grid_meta = prepare_strip_grid(
    df_all,
    df_plot,
    row_variable=row_variable,
    fixed_filters=fixed_filters_grid,
    colour_hex_col=colour_hex_col,
    x="design_peak_wavelength_nm",
    score_col="eta_C",
    select_best_within_cell=True,
)

# Optional info messages
if grid_meta["removed_filter"] is not None:
    print(
        f"Removed '{row_variable}={grid_meta['removed_filter']}' from fixed_filters "
        f"because the row variable must vary across rows."
    )

print("Fixed filters used:")
print(grid_meta["fixed_filters_clean"])
print(f"Row variable : {grid_meta['row_variable']}")
print(f"Colour column: {grid_meta['colour_hex_col']}")
print(f"Source df    : {grid_meta['df_source_name']}")
print(f"Selected tile cases: {len(df_grid)}")

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, ax = plot_colour_strip_grid(
    df_grid,
    row_variable=grid_meta["row_variable"],
    x="design_peak_wavelength_nm",
    colour_hex_col=grid_meta["colour_hex_col"],
    annotate_cols=["eta_C", "eta_L"],
    row_label_fmt=grid_meta["row_label_fmt"],
    title=grid_meta["title"],
    overlay_line_cols=["eta_C", "eta_L"],
    overlay_line_colors={"eta_C": "black", "eta_L": "white"},
    overlay_line_scale="global_shared",
    overlay_reference_values=[0.0, 1.0],
)

plt.show()

# ------------------------------------------------------------
# Compact summary table of the plotted tiles
# ------------------------------------------------------------
df_grid_print = summarize_strip_grid_cases(
    df_grid,
    row_variable=grid_meta["row_variable"],
    fixed_filters=grid_meta["fixed_filters_clean"],
    colour_hex_col=grid_meta["colour_hex_col"],
    x="design_peak_wavelength_nm",
)

display(df_grid_print.sort_values("design_peak_wavelength_nm"))

In [ ]:
# (1.2) CIELAB a*–b* plane
plt.figure(figsize=(6,6))
plt.scatter(df_all["a"], df_all["b"], c=df_all["hex"], s=15)
plt.axhline(0, color="gray")
plt.axvline(0, color="gray")
plt.xlabel("a*")
plt.ylabel("b*")
plt.title("CIELAB a*–b* plane")
plt.axis("equal")
plt.show()

CELL 6 — η_C AND η_L HEATMAPS (1)

In [ ]:
subset = df_all[
    (df_all["n_A"] == 2.1) &
    (df_all["k"] == 0.05) &
    (np.isclose(df_all["collection_angle_deg"], 90.0))
]

from bragg_onion.sweep import plot_sweep_heatmap

plot_sweep_heatmap(
    subset,
    x="design_peak_wavelength_nm",
    y="n_layers",
    value="eta_C",
)
plt.title("η_C heatmap")
plt.show()

plot_sweep_heatmap(
    subset,
    x="design_peak_wavelength_nm",
    y="n_layers",
    value="eta_L",
)
plt.title("η_L heatmap")
plt.show()

CELL 7 — SCATTERING ANALYSIS (2.0)

In [ ]:
from bragg_onion._notebook_tools.case_selection import (
    select_single_case_by_filters,
)

filters = {
    "k": 0.05,
    "n_layers": 100,
    "design_peak_wavelength_nm": 560.0,
    "collection_angle_deg": 90.0,
    "n_A": 1.8,
}

row_case = select_single_case_by_filters(df_plot, filters)
display(row_case)

In [ ]:
from bragg_onion._notebook_tools.scattering_extract import (
    rerun_single_case,
    extract_case_arrays,
)

analysis_wavelengths_nm = np.arange(380, 781, 1)
analysis_theta_deg = np.linspace(0, 180, 721)

sweep_case = rerun_single_case(
    row_case,
    analysis_wavelengths_nm,
    analysis_theta_deg,
)

data = extract_case_arrays(sweep_case)

In [ ]:
from bragg_onion._notebook_tools.scattering_plots import plot_differential_vs_theta

plot_differential_vs_theta(
    data["theta_deg"],
    data["wavelengths_nm"],
    data["dcs_geom"],
    plot_all=True,
    y_mode="db"
)

In [ ]:
import bragg_onion._notebook_tools.scattering_plots
from importlib import reload
reload(bragg_onion._notebook_tools.scattering_plots)
from bragg_onion._notebook_tools.scattering_plots import (
    plot_polar_differential_scattering
)

plot_polar_differential_scattering(
    data["theta_deg"],
    data["wavelengths_nm"],
    data["dcs_geom"],
    plot_all=True,
    y_mode="db",
)

plot_polar_differential_scattering(
    data["theta_deg"],
    data["wavelengths_nm"],
    data["dcs_geom"],
    plot_all=False,
    selected_wavelengths_nm=[450, 550, 650],
    show_legend=True,
    y_mode="db",
)

In [ ]:
import bragg_onion._notebook_tools.scattering_sectors
from importlib import reload
reload(bragg_onion._notebook_tools.scattering_sectors)
from bragg_onion._notebook_tools.scattering_sectors import (
    integrate_sectors,
    plot_scattering_sectors,
)

sector = integrate_sectors(
    np.deg2rad(data["theta_deg"]),
    data["dcs_geom"],
    alpha_deg=70.5,
)

plot_scattering_sectors(
    data["wavelengths_nm"],
    sector,
    alpha_deg=70.5,
    y_mode="db",
)

In [ ]:
from bragg_onion._notebook_tools.scattering_sectors import plot_scattering_pie


plot_scattering_pie(
    data["wavelengths_nm"],
    sector,
    wavelength_nm=row_case["design_peak_wavelength_nm"],
    alpha_deg=70.5,
    include_absorption=True,
    qsca=data["qsca"],
    qabs=data["qabs"],
)

In [ ]:
from bragg_onion._notebook_tools.scattering_plots import (
    plot_integrated_efficiencies
)

plot_integrated_efficiencies(
    data["wavelengths_nm"],
    qsca=data["qsca"],
    qext=data["qext"],
    qabs=data["qabs"],
    wavelengths_int_nm=data["wavelengths_int_nm"],
    q_collected_geom=data["q_collected_geom"],
    row_case=row_case,
)

In [ ]:
from bragg_onion._notebook_tools.scattering_plots import (
    plot_integrated_with_g_albedo
)

plot_integrated_with_g_albedo(
    data,
    row_case=row_case,
    integ=integ,
    show_diagnostics=True,
)

In [ ]:
import bragg_onion._notebook_tools.scattering_sectors
from importlib import reload
reload(bragg_onion._notebook_tools.scattering_sectors)
from bragg_onion._notebook_tools.scattering_sectors import (
    plot_backward_alpha_sweep,
)

plot_backward_alpha_sweep(
    np.deg2rad(data["theta_deg"]),
    data["dcs_geom"],
    data["wavelengths_nm"],
    alpha_list=[10, 30, 50, 70.5, 90.0],
    row_case=row_case,
    normalize=True,
)


CELL 8 — NEAR‑FIELD ANALYSIS (2.1)

In [ ]:
# Choose wavelengths & plane for field inspection
# Wavelengths for near-field inspection
field_wavelengths_nm = [
    float(row_case["design_peak_wavelength_nm"]),
]

# Optional: add colour-driving peak wavelength if available
# if "q_collected_geom_peak_wavelength_nm" in row_case:
#     field_wavelengths_nm.append(
#         float(row_case["q_collected_geom_peak_wavelength_nm"])
#     )

# Choose plane and resolution
FIELD_PLANE = "xz"     # "xy", "xz", or "yz"
FIELD_NPTS = 501
EXTENT_FACTOR = 1   # extent = factor × outer radius
print(field_wavelengths_nm)

In [ ]:
from bragg_onion.materials import ConstantDispersion

mat_A = ConstantDispersion.from_nk(
    "A",
    n=float(row_case["n_A"]),
    k=float(row_case["k"]),
)

mat_B = ConstantDispersion.from_nk(
    "B",
    n=float(row_case["n_B"]),
    k=float(row_case["k"]),
)

n_medium=1.0

In [ ]:
from bragg_onion._notebook_tools.field_analysis import rebuild_stack_from_row

FIXED_GEOMETRY_KWARGS = {}

geometry_sel, stack_sel = rebuild_stack_from_row(
    row_case,
    material_a=mat_A,
    material_b=mat_B,
    fixed_geometry_kwargs=FIXED_GEOMETRY_KWARGS,
)

In [ ]:
# Compute field maps
from bragg_onion.fields import compute_field_map

field_maps = {}

for wl_nm in field_wavelengths_nm:
    print(f"Computing field map at {wl_nm:.1f} nm")

    field_maps[wl_nm] = compute_field_map(
        stack=stack_sel,
        wavelength_m=wl_nm * 1e-9,
        n_medium=n_medium,
        plane=FIELD_PLANE,
        npts=FIELD_NPTS,
        extent_outer_radius_factor=EXTENT_FACTOR,
    )

print("Field maps computed:", list(field_maps.keys()))

In [ ]:
# field magnitude plots (|E|, |H|, |S|)
# A) Total field magnitude |E|
import bragg_onion.fields
from importlib import reload
reload(bragg_onion.fields)
from bragg_onion.fields import plot_field_magnitude

for wl_nm, fm in field_maps.items():
    plot_field_magnitude(
        fm,
        quantity="E",
        kind="total",
        scale="db",
        clip_percentile_low=0,
        clip_percentile_high=85,
        cmap="turbo", # viridis
    )
    plt.show()

# B) Scattered‑like |E| (very informative!)
for wl_nm, fm in field_maps.items():
    plot_field_magnitude(
        fm,
        quantity="E",
        kind="scattered",
        scale="db",
        clip_percentile_low=0,
        clip_percentile_high=81,
        cmap="turbo", # magma
    )
    plt.show()

#  |S| magnitude (total energy flow)
for wl_nm, fm in field_maps.items():
    plot_field_magnitude(
        fm,
        quantity="S",
        kind="total",
        scale="db",
        floor = 1e-30,
        clip_percentile_low=0,
        clip_percentile_high=100,
        cmap="turbo", # inferno
    )
    plt.show()

In [ ]:
# Poynting streamlines
# A) Delta flow
from bragg_onion.fields import plot_poynting_streamlines

for wl_nm, fm in field_maps.items():
    plot_poynting_streamlines(
        fm,
        flow_kind="delta",
        background_quantity="S",
        background_kind="delta",
        background_scale="db",
        # background_floor=1e-14,
        # background_cmap="inferno",
        background_clip_percentile_high=99.7,
        streamline_density=1.4,
        streamline_color="white",
        # streamline_linewidth=0.9,
        normalize_vectors=True,
        min_speed_fraction=0,
        mask_inside_sphere=False,
    )
    plt.show()

# B) Total flow (reference)
for wl_nm, fm in field_maps.items():
    plot_poynting_streamlines(
        fm,
        flow_kind="total",
        background_quantity="S",
        background_kind="total",
        background_scale="db",
        background_clip_percentile_high=99.7,
        streamline_density=1.4,
        streamline_color="white",
        normalize_vectors=False,
        min_speed_fraction=0.00,
        mask_inside_sphere=False,
    )
    plt.show()

# C) Scattered‑like flow
for wl_nm, fm in field_maps.items():
    plot_poynting_streamlines(
        fm,
        flow_kind="scattered",
        background_quantity="S",
        background_kind="scattered",
        background_scale="db",
        background_clip_percentile_high=99.7,
        streamline_density=1.4,
        streamline_color="white",
        normalize_vectors=True,
        min_speed_fraction=0.00,
        mask_inside_sphere=False,
    )
    plt.show()

CELL 9 — MIE RESONANCES (2.2)

You already implemented this in Point 6:

- l‑resolved bar plots

- l‑resolved spectra

✅ reuse those exact cells here for:

- top cases

- representative parameter cuts

In [ ]:
# CELL 10 — SENSITIVITY ANALYSIS (OPTIONAL, RECOMMENDED)
# Example: η_C vs n_layers for different n_A.
plt.figure(figsize=(7,4.5))

for n_A in n_A_vals:
    subset = df_all[
        (df_all["n_A"] == n_A) &
        (df_all["k"] == 0.05) &
        (np.isclose(df_all["collection_angle_deg"], 70.5))
    ]
    grp = subset.groupby("n_layers")["eta_C"].mean()
    plt.plot(grp.index, grp.values, marker="o", label=f"n_A={n_A:.2f}")

plt.xlabel("Number of layers")
plt.ylabel("η_C")
plt.legend()
plt.title("Sensitivity of η_C to n_layers and refractive index contrast")
plt.show()